In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

In [2]:
DATA_PATH = "../data/processed/processed_transactions.csv"

import pandas as pd

df = pd.read_csv(DATA_PATH)

df.head()

,transaction_id,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,...,abs_destination_balance_error,amount_to_oldbalanceOrg_ratio,amount_to_oldbalanceDest_ratio,is_zero_oldbalanceOrg,is_zero_oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,...,9839.64,0.057834,9839.640000,0,1,0,0,0,1,0
1,2,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,...,1864.28,0.087731,1864.280000,0,1,0,0,0,1,0
2,3,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,...,181.00,0.994505,181.000000,0,1,0,0,0,0,1
3,4,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,...,21363.00,0.994505,0.008545,0,0,0,1,0,0,0
4,5,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,...,11668.14,0.280788,11668.140000,0,1,0,0,0,1,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 28 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   transaction_id                  int64  
 1   step                            int64  
 2   type                            str    
 3   amount                          float64
 4   oldbalanceOrg                   float64
 5   newbalanceOrig                  float64
 6   oldbalanceDest                  float64
 7   newbalanceDest                  float64
 8   isFraud                         int64  
 9   isFlaggedFraud                  int64  
 10  is_origin_customer              int64  
 11  is_dest_customer                int64  
 12  is_dest_merchant                int64  
 13  origin_balance_diff             float64
 14  destination_balance_diff        float64
 15  origin_balance_error            float64
 16  destination_balance_error       float64
 17  abs_origin_balance_error        float6

In [4]:
df.describe()

,transaction_id,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,is_origin_customer,...,abs_destination_balance_error,amount_to_oldbalanceOrg_ratio,amount_to_oldbalanceDest_ratio,is_zero_oldbalanceOrg,is_zero_oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6362620.0,...,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,3.181310e+06,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06,1.0,...,9.359907e+04,7.067448e+04,2.788383e+04,3.304376e-01,4.250431e-01,2.199226e-01,3.516633e-01,6.511783e-03,3.381461e-01,8.375622e-02
std,1.836730e+06,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03,0.0,...,4.350570e+05,5.084243e+05,1.879306e+05,4.703707e-01,4.943496e-01,4.141940e-01,4.774895e-01,8.043246e-02,4.730786e-01,2.770219e-01
min,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.590656e+06,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,...,0.000000e+00,2.344011e-01,1.607037e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,3.181310e+06,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00,1.0,...,5.123620e+03,6.453832e+00,9.178313e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,4.771965e+06,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00,1.0,...,4.342133e+04,1.228776e+04,9.732300e+03,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00
max,6.362620e+06,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00,1.0,...,7.588573e+07,9.244552e+07,6.096528e+07,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [5]:
df.isnull().sum()

transaction_id                    0
step                              0
type                              0
amount                            0
oldbalanceOrg                     0
newbalanceOrig                    0
oldbalanceDest                    0
newbalanceDest                    0
isFraud                           0
isFlaggedFraud                    0
is_origin_customer                0
is_dest_customer                  0
is_dest_merchant                  0
origin_balance_diff               0
destination_balance_diff          0
origin_balance_error              0
destination_balance_error         0
abs_origin_balance_error          0
abs_destination_balance_error     0
amount_to_oldbalanceOrg_ratio     0
amount_to_oldbalanceDest_ratio    0
is_zero_oldbalanceOrg             0
is_zero_oldbalanceDest            0
type_CASH_IN                      0
type_CASH_OUT                     0
type_DEBIT                        0
type_PAYMENT                      0
type_TRANSFER               

In [6]:
# Drop unnecessary columns
cols_to_drop=[c for c in ['transaction_id','type','isFlaggedFraud','is_origin_customer'] if c in df.columns]
df=df.drop(columns=cols_to_drop)
print('Dropped:',cols_to_drop)
df.head()

Dropped: ['transaction_id', 'type', 'isFlaggedFraud', 'is_origin_customer']


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,is_dest_customer,is_dest_merchant,origin_balance_diff,...,abs_destination_balance_error,amount_to_oldbalanceOrg_ratio,amount_to_oldbalanceDest_ratio,is_zero_oldbalanceOrg,is_zero_oldbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,1,9839.64,...,9839.64,0.057834,9839.640000,0,1,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,1,1864.28,...,1864.28,0.087731,1864.280000,0,1,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,1,0,181.00,...,181.00,0.994505,181.000000,0,1,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,1,0,181.00,...,21363.00,0.994505,0.008545,0,0,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,1,11668.14,...,11668.14,0.280788,11668.140000,0,1,0,0,0,1,0


In [7]:
TARGET='isFraud'
X=df.drop(columns=[TARGET])
y=df[TARGET]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,confusion_matrix
import pandas as pd
models={
'Logistic Regression':Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))]),
'Decision Tree':DecisionTreeClassifier(random_state=42,class_weight='balanced'),
'Random Forest':RandomForestClassifier(n_estimators=200,random_state=42,class_weight='balanced',n_jobs=-1),
'HistGradientBoosting':HistGradientBoostingClassifier(random_state=42)
}
results=[];trained={}
for n,m in models.items():
 m.fit(X_train,y_train);p=m.predict(X_test);cm=confusion_matrix(y_test,p);fn=cm[1][0];
 results.append({'Model':n,'Precision':precision_score(y_test,p),'Recall':recall_score(y_test,p),'F1':f1_score(y_test,p),'False Negatives':int(fn)});trained[n]=m;print(n);print(cm)
results_df=pd.DataFrame(results).sort_values(['Recall','F1'],ascending=False);results_df

Logistic Regression
[[1196226   74655]
 [     35    1608]]
Decision Tree
[[1270877       4]
 [      4    1639]]
Random Forest
[[1270880       1]
 [      4    1639]]
HistGradientBoosting
[[1270860      21]
 [      6    1637]]


,Model,Precision,Recall,F1,False Negatives
2,Random Forest,0.999390,0.997565,0.998477,4
1,Decision Tree,0.997565,0.997565,0.997565,4
3,HistGradientBoosting,0.987334,0.996348,0.991821,6
0,Logistic Regression,0.021085,0.978698,0.041281,35


In [9]:
best_name=results_df.iloc[0]['Model']
best_model=trained[best_name]
print('Best:',best_name)
results_df

Best: Random Forest


,Model,Precision,Recall,F1,False Negatives
2,Random Forest,0.999390,0.997565,0.998477,4
1,Decision Tree,0.997565,0.997565,0.997565,4
3,HistGradientBoosting,0.987334,0.996348,0.991821,6
0,Logistic Regression,0.021085,0.978698,0.041281,35


In [10]:
import json,joblib,os
os.makedirs('../models/saved_model',exist_ok=True)
joblib.dump(best_model,'../models/saved_model/fraud_detection_model.pkl')
joblib.dump(list(X.columns),'../models/saved_model/model_features.pkl')
meta={'best_model':best_name,'results':results_df.to_dict(orient='records')}
with open('../models/saved_model/model_metadata.json','w') as f: json.dump(meta,f,indent=2)
print('Saved.')

Saved.
